In [1]:
from pathlib import Path

# 你在 notebooks/ 里运行，所以 ROOT 是上一级
ROOT = Path.cwd().resolve().parents[0]
print("CWD :", Path.cwd())
print("ROOT:", ROOT)

PKL_PATH = ROOT / "data" / "simulated_data_dict.pkl"
FEAT_PATH = ROOT / "data" / "sim_features.parquet"

MODEL_PATH = ROOT / "models" / "pattern_ovr_8class_full_80_20.joblib"
OUT_PARQ   = ROOT / "data" / "sim_features_with_pattern_8class_full_80_20.parquet"

print("PKL exists :", PKL_PATH.exists(), PKL_PATH)
print("FEAT exists:", FEAT_PATH.exists(), FEAT_PATH)

CWD : /home/lixiangyu/xwq/Subcellular_Feature/notebooks
ROOT: /home/lixiangyu/xwq/Subcellular_Feature
PKL exists : True /home/lixiangyu/xwq/Subcellular_Feature/data/simulated_data_dict.pkl
FEAT exists: True /home/lixiangyu/xwq/Subcellular_Feature/data/sim_features.parquet


读取模拟数据的groundtruth+读取17特征并Merge

In [2]:
import pickle
import pandas as pd
import numpy as np

obj = pickle.load(open(PKL_PATH, "rb"))
spot = obj["data_df"].copy()         # 点级别，含 groundtruth
feat = pd.read_parquet(FEAT_PATH).copy()   # cell-gene级别 17 特征

# groundtruth 聚合到 cell-gene
y_df = (
    spot.groupby(["cell", "gene"], as_index=False)["groundtruth"]
    .first()
    .rename(columns={"groundtruth": "y_true"})
)

# 统一类型，避免 merge 错位
for c in ["cell", "gene"]:
    y_df[c] = y_df[c].astype(str).str.strip()
    feat[c] = feat[c].astype(str).str.strip()

data = feat.merge(y_df, on=["cell", "gene"], how="inner")

print("spot :", spot.shape)
print("feat :", feat.shape)
print("merged:", data.shape)
print("\nlabel dist:\n", data["y_true"].value_counts())
data.head()

spot : (1347881, 12)
feat : (20000, 19)
merged: (20000, 20)

label dist:
 y_true
Random          2500
Foci            2500
Nuclear         2500
Cytoplasmic     2500
Nuclear edge    2500
Cell edge       2500
Protrusion      2500
Radial          2500
Name: count, dtype: int64


,cell,gene,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,bento_nucleus_dispersion_norm,bento_l_max,bento_l_max_gradient,bento_l_min_gradient,bento_l_monotony,bento_l_half_radius,sprawl_peripheral,sprawl_central,sprawl_punctate,sprawl_radial,y_true
0,cell_102,gene_0_0_0,0.650084,0.786023,0.754027,0.562135,0.316887,0.789136,0.664003,0.799773,3.076283,1.012939,-0.880864,-0.937887,-2.462147,-0.173893,0.163147,0.46,0.70,Random
1,cell_102,gene_0_0_1,0.654681,0.823567,0.698191,0.617036,0.362271,0.853534,0.868480,0.832679,3.255770,0.579378,-0.708565,-0.942784,-5.598550,-0.150597,0.183738,0.10,-0.83,Random
2,cell_102,gene_0_0_2,0.661598,0.820110,0.736471,0.571356,0.356906,0.803288,0.648734,0.634645,3.345068,0.428725,-0.758260,-0.934486,-2.227309,-0.158413,0.177275,-0.62,-0.91,Random
3,cell_102,gene_0_0_3,0.667682,0.832543,0.755766,0.589740,0.368915,0.789935,0.739277,0.689100,0.964089,0.240149,-0.761459,-0.939861,-5.406958,-0.088825,0.175810,-0.70,0.03,Random
4,cell_102,gene_0_0_4,0.657550,0.804136,0.713734,0.591729,0.345639,0.833883,0.765556,0.719655,0.736492,0.289656,-0.724002,-0.960981,-5.511314,-0.140678,0.146013,0.68,0.05,Random


In [3]:
CLASSES = [
    "Foci",
    "Nuclear",
    "Cytoplasmic",
    "Nuclear edge",
    "Cell edge",
    "Protrusion",
    "Radial",
    "Random",
]

FEATURE_COLS = [
    c for c in data.columns
    if c not in ("cell", "gene", "y_true")
]

print("n_features:", len(FEATURE_COLS))
print(FEATURE_COLS)

n_features: 17
['bento_cell_inner_proximity', 'bento_nucleus_inner_proximity', 'bento_nucleus_outer_proximity', 'bento_cell_inner_asymmetry', 'bento_nucleus_inner_asymmetry', 'bento_nucleus_outer_asymmetry', 'bento_point_dispersion_norm', 'bento_nucleus_dispersion_norm', 'bento_l_max', 'bento_l_max_gradient', 'bento_l_min_gradient', 'bento_l_monotony', 'bento_l_half_radius', 'sprawl_peripheral', 'sprawl_central', 'sprawl_punctate', 'sprawl_radial']


80/20分割，按cell分组


In [4]:
from sklearn.model_selection import GroupShuffleSplit

X = data[FEATURE_COLS].copy()
y = data["y_true"].astype(str).copy()
groups = data["cell"].astype(str).values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_train_all = X.iloc[tr_idx].reset_index(drop=True)
y_train_all = y.iloc[tr_idx].reset_index(drop=True)
cell_train_all = data["cell"].iloc[tr_idx].reset_index(drop=True)

X_test = X.iloc[te_idx].reset_index(drop=True)
y_test = y.iloc[te_idx].reset_index(drop=True)
cell_test = data["cell"].iloc[te_idx].reset_index(drop=True)

print("train:", X_train_all.shape, "test:", X_test.shape)
print("train cells:", cell_train_all.nunique(), "test cells:", cell_test.nunique())

train: (16000, 17) test: (4000, 17)
train cells: 40 test cells: 10


训练集内部切分验证集，用于阈值校准

In [5]:
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
tr2, val2 = next(gss2.split(X_train_all, y_train_all, groups=cell_train_all))

X_subtrain = X_train_all.iloc[tr2].reset_index(drop=True)
y_subtrain = y_train_all.iloc[tr2].reset_index(drop=True)
cell_subtrain = cell_train_all.iloc[tr2].reset_index(drop=True)

X_val = X_train_all.iloc[val2].reset_index(drop=True)
y_val = y_train_all.iloc[val2].reset_index(drop=True)
cell_val = cell_train_all.iloc[val2].reset_index(drop=True)

print("subtrain:", X_subtrain.shape, "val:", X_val.shape)
print("subtrain cells:", cell_subtrain.nunique(), "val cells:", cell_val.nunique())
print("\nsubtrain label dist:\n", y_subtrain.value_counts())
print("\nval label dist:\n", y_val.value_counts())

subtrain: (12800, 17) val: (3200, 17)
subtrain cells: 32 val cells: 8

subtrain label dist:
 y_true
Random          1600
Foci            1600
Nuclear         1600
Cytoplasmic     1600
Nuclear edge    1600
Cell edge       1600
Protrusion      1600
Radial          1600
Name: count, dtype: int64

val label dist:
 y_true
Random          400
Foci            400
Nuclear         400
Cytoplasmic     400
Nuclear edge    400
Cell edge       400
Protrusion      400
Radial          400
Name: count, dtype: int64


训练8个OVR二分类器+概率校准

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

def make_binary_model(random_state=0):
    base = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=400,
            n_jobs=-1,
            random_state=random_state,
            class_weight=None
        ))
    ])
    # 概率校准：sigmoid 通常更稳更快
    return CalibratedClassifierCV(base, cv=3, method="sigmoid")

models = {}
for i, c in enumerate(CLASSES):
    y_bin = (y_subtrain == c).astype(int).values
    m = make_binary_model(random_state=100 + i)
    m.fit(X_subtrain, y_bin)
    models[c] = m

print("trained:", list(models.keys()))

trained: ['Foci', 'Nuclear', 'Cytoplasmic', 'Nuclear edge', 'Cell edge', 'Protrusion', 'Radial', 'Random']


在验证集上自动找每类阈值

In [7]:
from sklearn.metrics import f1_score

def best_threshold(probs, y_true_bin, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    best_t, best_f1 = 0.5, -1
    for t in grid:
        pred = (probs >= t).astype(int)
        f1 = f1_score(y_true_bin, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

thresholds = {}
val_f1 = {}

for c in CLASSES:
    p = models[c].predict_proba(X_val)[:, 1]
    yb = (y_val == c).astype(int).values
    t, f1 = best_threshold(p, yb)
    thresholds[c] = t
    val_f1[c] = f1

print("thresholds:", thresholds)
print("val_f1:", val_f1)

thresholds: {'Foci': 0.1, 'Nuclear': 0.15, 'Cytoplasmic': 0.35, 'Nuclear edge': 0.5499999999999999, 'Cell edge': 0.3, 'Protrusion': 0.5499999999999999, 'Radial': 0.2, 'Random': 0.35}
val_f1: {'Foci': 0.9281045751633987, 'Nuclear': 0.9849624060150376, 'Cytoplasmic': 0.782608695652174, 'Nuclear edge': 0.9326556543837357, 'Cell edge': 0.9230769230769231, 'Protrusion': 0.9975062344139651, 'Radial': 0.9525, 'Random': 0.8369829683698297}


定义推理规则（8类multi-label+top1）
  pattern_multi:所有过阈值的类（无则选概率最大）   如有其余七类则去掉random
  pattern_top1:8类中概率最大


In [8]:
def predict_prob_df(X_in: pd.DataFrame) -> pd.DataFrame:
    probs = {c: models[c].predict_proba(X_in)[:, 1] for c in CLASSES}
    return pd.DataFrame(probs)

def infer_multi_top1(prob_df: pd.DataFrame):
    multi = []
    top1 = []

    for i in range(len(prob_df)):
        row = prob_df.iloc[i]

        # multi-label: 过阈值的类都选
        labels = [c for c in CLASSES if row[c] >= thresholds[c]]

        # 若一个都没过阈值，兜底取概率最大类
        if len(labels) == 0:
            labels = [row.idxmax()]

        # 若 Random 和别的类同时命中，则去掉 Random
        if "Random" in labels and len(labels) > 1:
            labels = [c for c in labels if c != "Random"]

        multi.append(labels)

        # top1: 直接8类中概率最大者
        top1.append(row.idxmax())

    return multi, top1

在测试集上评估效果

In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

prob_test = predict_prob_df(X_test)
multi_test, top1_test = infer_multi_top1(prob_test)

acc = accuracy_score(y_test, top1_test)
hit_multi = np.mean([gt in pm for gt, pm in zip(y_test.tolist(), multi_test)])

print("TEST top1 acc:", acc)
print("TEST multi-hit:", hit_multi)

print("\nReport (TEST top1):")
print(classification_report(y_test, top1_test, labels=CLASSES, digits=4))

cm = confusion_matrix(y_test, top1_test, labels=CLASSES)
cm_df = pd.DataFrame(
    cm,
    index=[f"true:{c}" for c in CLASSES],
    columns=[f"pred:{c}" for c in CLASSES]
)
cm_df

TEST top1 acc: 0.92375
TEST multi-hit: 0.92675

Report (TEST top1):
              precision    recall  f1-score   support

        Foci     0.9747    0.9240    0.9487       500
     Nuclear     0.9841    0.9920    0.9880       500
 Cytoplasmic     0.8241    0.8340    0.8290       500
Nuclear edge     0.9004    0.9040    0.9022       500
   Cell edge     0.9142    0.9800    0.9459       500
  Protrusion     0.9940    0.9940    0.9940       500
      Radial     0.9874    0.9440    0.9652       500
      Random     0.8180    0.8180    0.8180       500

    accuracy                         0.9237      4000
   macro avg     0.9246    0.9237    0.9239      4000
weighted avg     0.9246    0.9237    0.9239      4000



,pred:Foci,pred:Nuclear,pred:Cytoplasmic,pred:Nuclear edge,pred:Cell edge,pred:Protrusion,pred:Radial,pred:Random
true:Foci,462,0,19,5,2,3,0,9
true:Nuclear,0,496,0,1,0,0,0,3
true:Cytoplasmic,11,0,417,8,21,0,0,43
true:Nuclear edge,0,1,7,452,0,0,4,36
true:Cell edge,0,0,9,0,490,0,1,0
true:Protrusion,0,0,0,0,2,497,1,0
true:Radial,0,0,4,3,21,0,472,0
true:Random,1,7,50,33,0,0,0,409


保存模型joblib+全量预测parquet

In [10]:
import joblib

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

save_obj = {
    "models": models,
    "thresholds": thresholds,
    "feature_cols": FEATURE_COLS,
    "classes": CLASSES,
    "meta": {
        "split": "GroupShuffleSplit by cell (80/20), val from train (80/20)",
        "random_state_split": 42,
        "random_state_valsplit": 123,
        "train_cells": sorted(cell_train_all.astype(str).unique().tolist()),
        "test_cells": sorted(cell_test.astype(str).unique().tolist()),
        "val_cells": sorted(cell_val.astype(str).unique().tolist()),
        "val_f1": val_f1,
        "test_top1_acc": float(acc),
        "test_multi_hit": float(hit_multi),
    }
}
joblib.dump(save_obj, MODEL_PATH)
print("saved model:", MODEL_PATH)

# 全量预测（用于实际应用）
prob_all = predict_prob_df(X.reset_index(drop=True))
multi_all, top1_all = infer_multi_top1(prob_all)

out = data[["cell", "gene", "y_true"]].copy().reset_index(drop=True)

# 保存8类概率
for c in CLASSES:
    out[f"p_{c}"] = prob_all[c].values

out["pattern_multi"] = multi_all
out["pattern_top1"] = top1_all

OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(OUT_PARQ, index=False)
print("saved parquet:", OUT_PARQ)

out.head()

saved model: /home/lixiangyu/xwq/Subcellular_Feature/models/pattern_ovr_8class_full_80_20.joblib
saved parquet: /home/lixiangyu/xwq/Subcellular_Feature/data/sim_features_with_pattern_8class_full_80_20.parquet


,cell,gene,y_true,p_Foci,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_multi,pattern_top1
0,cell_102,gene_0_0_0,Random,0.005169,0.002571,0.045729,0.009688,0.00346,0.003811,0.003243,0.879331,[Random],Random
1,cell_102,gene_0_0_1,Random,0.004810,0.002571,0.162257,0.008956,0.00346,0.003811,0.003201,0.704238,[Random],Random
2,cell_102,gene_0_0_2,Random,0.004810,0.002571,0.015014,0.009978,0.00346,0.003811,0.003255,0.985572,[Random],Random
3,cell_102,gene_0_0_3,Random,0.004962,0.002571,0.025912,0.009168,0.00346,0.003811,0.003201,0.976062,[Random],Random
4,cell_102,gene_0_0_4,Random,0.005029,0.002622,0.034442,0.008844,0.00346,0.003811,0.003201,0.971948,[Random],Random


逐一比较错分样本

In [11]:
# 全量错分
wrong_all = out[out["y_true"] != out["pattern_top1"]][
    ["cell", "gene", "y_true", "pattern_top1", "pattern_multi"]
].copy()

print("ALL wrong:", len(wrong_all), "/", len(out), "acc:", 1 - len(wrong_all) / len(out))
wrong_all.head(20)

ALL wrong: 582 / 20000 acc: 0.9709


,cell,gene,y_true,pattern_top1,pattern_multi
405,cell_11,gene_0_1_0,Random,Nuclear,[Nuclear]
411,cell_11,gene_0_2_1,Random,Nuclear edge,[Random]
415,cell_11,gene_0_3_0,Random,Nuclear edge,[Nuclear edge]
425,cell_11,gene_0_5_0,Random,Cytoplasmic,[Cytoplasmic]
430,cell_11,gene_0_6_0,Random,Cytoplasmic,[Cytoplasmic]
433,cell_11,gene_0_6_3,Random,Cytoplasmic,[Cytoplasmic]
437,cell_11,gene_0_7_2,Random,Cytoplasmic,[Cytoplasmic]
438,cell_11,gene_0_7_3,Random,Nuclear edge,[Nuclear edge]
442,cell_11,gene_0_8_2,Random,Cytoplasmic,[Cytoplasmic]
445,cell_11,gene_0_9_0,Random,Cytoplasmic,[Cytoplasmic]


In [12]:
test_out = data.iloc[te_idx][["cell", "gene", "y_true"]].reset_index(drop=True).copy()

for c in CLASSES:
    test_out[f"p_{c}"] = prob_test[c].values

test_out["pattern_multi"] = multi_test
test_out["pattern_top1"] = top1_test

wrong_test = test_out[test_out["y_true"] != test_out["pattern_top1"]][
    ["cell", "gene", "y_true", "pattern_top1", "pattern_multi"]
    + [f"p_{c}" for c in CLASSES]
].copy()

print("TEST wrong:", len(wrong_test), "/", len(test_out), "acc:", 1 - len(wrong_test) / len(test_out))
wrong_test.head(30)

TEST wrong: 305 / 4000 acc: 0.92375


,cell,gene,y_true,pattern_top1,pattern_multi,p_Foci,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random
6,cell_167,gene_0_1_1,Random,Cytoplasmic,[Cytoplasmic],0.005149,0.002571,0.945457,0.013517,0.003460,0.003811,0.003201,0.024992
9,cell_167,gene_0_1_4,Random,Cytoplasmic,[Cytoplasmic],0.004978,0.002571,0.952087,0.008844,0.003525,0.003811,0.003201,0.032765
10,cell_167,gene_0_2_0,Random,Cytoplasmic,[Cytoplasmic],0.006556,0.002571,0.839586,0.008903,0.003510,0.003811,0.003201,0.088374
15,cell_167,gene_0_3_0,Random,Cytoplasmic,[Cytoplasmic],0.006654,0.002571,0.650379,0.009406,0.003460,0.003811,0.003201,0.073218
22,cell_167,gene_0_4_2,Random,Cytoplasmic,[Cytoplasmic],0.005274,0.002571,0.833815,0.008835,0.003481,0.003811,0.003201,0.223578
23,cell_167,gene_0_4_3,Random,Cytoplasmic,[Cytoplasmic],0.007377,0.002571,0.540993,0.013027,0.003460,0.003811,0.003201,0.365312
25,cell_167,gene_0_5_0,Random,Cytoplasmic,[Cytoplasmic],0.007402,0.002571,0.642525,0.011334,0.003460,0.003811,0.003222,0.093312
35,cell_167,gene_0_7_0,Random,Cytoplasmic,[Cytoplasmic],0.013534,0.002584,0.771242,0.017347,0.003837,0.003811,0.003243,0.026442
41,cell_167,gene_0_8_1,Random,Cytoplasmic,[Cytoplasmic],0.005156,0.002571,0.974620,0.008723,0.003460,0.003811,0.003201,0.018606
48,cell_167,gene_0_9_3,Random,Cytoplasmic,[Cytoplasmic],0.005297,0.002571,0.869697,0.008787,0.003460,0.003811,0.003201,0.243052


In [14]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm_all = confusion_matrix(
    out["y_true"],
    out["pattern_top1"],
    labels=CLASSES
)

cm_all_df = pd.DataFrame(
    cm_all,
    index=[f"true:{c}" for c in CLASSES],
    columns=[f"pred:{c}" for c in CLASSES]
)

cm_all_df

,pred:Foci,pred:Nuclear,pred:Cytoplasmic,pred:Nuclear edge,pred:Cell edge,pred:Protrusion,pred:Radial,pred:Random
true:Foci,2410,1,52,9,6,4,2,16
true:Nuclear,0,2487,0,1,0,0,0,12
true:Cytoplasmic,14,1,2339,18,39,0,0,89
true:Nuclear edge,0,1,8,2430,1,0,7,53
true:Cell edge,0,0,44,0,2454,0,2,0
true:Protrusion,0,0,0,0,2,2497,1,0
true:Radial,0,0,6,11,31,0,2449,3
true:Random,1,8,85,54,0,0,0,2352


In [16]:
# pattern_multi 里有多少样本包含多个标签
multi_len = out["pattern_multi"].apply(len)

n_multi_gt1 = (multi_len > 1).sum()
ratio_multi_gt1 = (multi_len > 1).mean()

print("pattern_multi 中标签数 > 1 的样本数:", n_multi_gt1)
print("pattern_multi 中标签数 > 1 的样本占比:", ratio_multi_gt1)

pattern_multi 中标签数 > 1 的样本数: 298
pattern_multi 中标签数 > 1 的样本占比: 0.0149


In [18]:
multi_more_than_one = out[out["pattern_multi"].apply(len) > 1].copy()

print("多标签样本数:", len(multi_more_than_one))
multi_more_than_one.head(20)

多标签样本数: 298


,cell,gene,y_true,p_Foci,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_multi,pattern_top1
160,cell_102,gene_3_2_0,Cytoplasmic,0.009910,0.219871,0.565797,0.205133,0.004141,0.003811,0.003707,0.047405,"[Nuclear, Cytoplasmic]",Cytoplasmic
165,cell_102,gene_3_3_0,Cytoplasmic,0.111360,0.002584,0.703263,0.008723,0.005615,0.003811,0.003577,0.015749,"[Foci, Cytoplasmic]",Cytoplasmic
195,cell_102,gene_3_9_0,Cytoplasmic,0.010011,0.002571,0.625659,0.008844,0.376765,0.003832,0.003375,0.011931,"[Cytoplasmic, Cell edge]",Cytoplasmic
465,cell_11,gene_1_3_0,Foci,0.268484,0.002596,0.519450,0.010810,0.018236,0.004027,0.003512,0.021606,"[Foci, Cytoplasmic]",Cytoplasmic
592,cell_11,gene_3_8_2,Cytoplasmic,0.004910,0.002571,0.691844,0.008723,0.483543,0.003811,0.003201,0.012108,"[Cytoplasmic, Cell edge]",Cytoplasmic
630,cell_11,gene_4_6_0,Nuclear edge,0.011761,0.003271,0.012760,0.774720,0.003814,0.003811,0.837665,0.013168,"[Nuclear edge, Radial]",Radial
700,cell_11,gene_6_0_0,Protrusion,0.460492,0.002571,0.019385,0.008787,0.005187,0.585091,0.003547,0.011850,"[Foci, Protrusion]",Protrusion
980,cell_12,gene_3_6_0,Cytoplasmic,0.045594,0.333858,0.515151,0.012732,0.003460,0.003811,0.009292,0.014452,"[Nuclear, Cytoplasmic]",Cytoplasmic
995,cell_12,gene_3_9_0,Cytoplasmic,0.408561,0.002571,0.648615,0.008956,0.003611,0.003811,0.004060,0.012495,"[Foci, Cytoplasmic]",Cytoplasmic
1100,cell_12,gene_6_0_0,Protrusion,0.159832,0.002571,0.012406,0.008723,0.007253,0.985886,0.003201,0.011850,"[Foci, Protrusion]",Protrusion


In [19]:
from collections import Counter

# 只保留多标签样本
multi_gt1 = out[out["pattern_multi"].apply(len) > 1].copy()

# 把列表转成可计数的 tuple
combo_counter = Counter(
    tuple(labels) for labels in multi_gt1["pattern_multi"]
)

# 出现次数从高到低
combo_df = (
    pd.DataFrame(
        combo_counter.items(),
        columns=["label_combo", "count"]
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

combo_df.head(20)

,label_combo,count
0,"(Foci, Cytoplasmic)",98
1,"(Cytoplasmic, Cell edge)",67
2,"(Nuclear edge, Radial)",46
3,"(Foci, Protrusion)",27
4,"(Foci, Cell edge)",11
5,"(Foci, Radial)",9
6,"(Cell edge, Radial)",9
7,"(Foci, Nuclear)",8
8,"(Nuclear, Radial)",8
9,"(Foci, Nuclear edge)",5
